### Step 1: Load the Silver dataset

In [0]:
silver_df = spark.table("workspace.ecommerce_project.silver_orders")

display(silver_df.limit(10))

order_id,customer_id,customer_name,customer_email,customer_phone,product_id,product_name,quantity,unit_price,discount_percent,order_amount,final_amount,order_date,ship_date,delivery_date,shipping_state,pincode,ingestion_timestamp,source_file,batch_id,source_system,pipeline_id,payment_status,order_status,payment_method,category,shipping_city,invalid_customer_name,invalid_customer_email,invalid_customer_phone,invalid_order_id,invalid_customer_id,invalid_product_id,invalid_product_name,invalid_category,invalid_quantity,invalid_unit_price,invalid_discount_percent,invalid_order_amount,invalid_final_amount,invalid_payment_method,invalid_payment_status,invalid_order_status,invalid_order_date,invalid_ship_date,invalid_delivery_date,invalid_ship_date_order,invalid_delivery_date_ship,invalid_shipped_date,invalid_deliver_date,invalid_placed_dates,invalid_confirmed_delivery,invalid_returned_date,payment_order_status_violation,invalid_shipping_city,invalid_shipping_state,invalid_pincode_format,invalid_pincode,duplicate_order,reject_reason,record_valid
ORD100218,CUST10020,Zara Sundaram,zara.sundaram@yahoo.com,8249926919,PROD1064,Cricket Bat,8,5037.33,0,40298.64,40298.64,2025-10-18,2025-10-20,null,Karnataka,560073,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Shipped,NET BANKING,SPORTS,Bengaluru,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100164,CUST10042,Mahesh Shetty,mahesh.shetty@rediffmail.com,7678998028,PROD1073,Mouse,4,2091.76,80,8367.04,1673.41,2025-12-13,null,null,Uttar Pradesh,226094,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,REFUNDED,Cancelled,NET BANKING,ELECTRONICS,Lucknow,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100064,CUST10136,Chitra Reddy,chitra.reddy@hotmail.com,8113585139,PROD1066,Books,4,2731.88,10,10927.52,9834.77,2025-11-27,null,null,Telangana,500032,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,REFUNDED,Cancelled,DEBIT CARD,BOOKS,Hyderabad,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100596,CUST10067,Kabir Siddiqui,kabir.siddiqui@gmail.com,8812308209,PROD1036,Water Bottle,9,334.78,80,3013.02,602.6,2025-09-06,2025-09-07,2025-09-08,Maharashtra,411008,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Delivered,UPI,HOME,Pune,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100660,CUST10016,Radha Chandran,radha.chandran@rediffmail.com,9389854268,PROD1048,Bluetooth Speaker,5,7784.98,10,38924.9,35032.41,2025-01-25,2025-01-26,2025-01-30,Rajasthan,302000,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Delivered,UPI,ELECTRONICS,Jaipur,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100071,CUST10105,Rajesh Siddiqui,rajesh.siddiqui@rediffmail.com,6698509918,PROD1100,Mixer Grinder,6,8724.76,0,52348.56,52348.56,2025-11-28,2025-12-01,2025-12-02,West Bengal,700036,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1,PAID,Delivered,WALLET,HOME,Kolkata,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100463,CUST10087,Vikram Murthy,vikram.murthy@yahoo.com,8141078352,PROD1049,LED Monitor,9,38853.83,30,349

### Step 2: Check the current populated partitions

In [0]:
from pyspark.sql.functions import spark_partition_id

print(
    "Populated partitions before coalesce:",
    silver_df
    .select(spark_partition_id().alias("partition_id"))
    .distinct()
    .count()
)

Populated partitions before coalesce: 2


### Step 3: Reduce the number of partitions

In [0]:
coalesced_df = silver_df.coalesce(1)

In [0]:
print(
    "Populated partition after coalesce:",
    coalesced_df
    .select(spark_partition_id().alias("partition_id"))
    .distinct()
    .count()
)

Populated partition after coalesce: 1


### Step 5: Compare the original and coalesced DataFrames

In [0]:
print("Original row count: ",silver_df.count())
print("Coalesced row count: ",coalesced_df.count())

Original row count:  738
Coalesced row count:  738


### Step 6: Write the result to a separate Delta table

In [0]:
(
    coalesced_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.ecommerce_project.silver_orders_coalesced")
)